# Object API examples

See the [Object API reference](reference.rst) for the list of available operator classes.


The class-based API mirrors `torchvision.transforms.v2`: instantiate an operator and call it.
Due to DALI's pipeline design, **operators must be wrapped inside `dtv.Compose`** — the `Compose`
object builds and owns the underlying DALI pipeline.

```python
pipeline = dtv.Compose(
    [
        dtv.Resize(size=(128, 128)),
        dtv.RandomHorizontalFlip(p=0.5),
        ...
    ]
)
output = pipeline(img)
```

`Compose` detects the input type automatically:
- `PIL.Image` → uses an HWC pipeline, returns `PIL.Image`
- `torch.Tensor` `(..., C, H, W)` → uses a pipeline whose layout depends on the number of dimensions and on `batch_size` (a 3-D tensor is `CHW`, see [Input layouts](../supported_data.ipynb#Input-layouts)), returns `torch.Tensor`

## Setup

Import the DALI TorchVision API (object-oriented as `dtv`, functional as `dtv_fn`) alongside the standard TorchVision counterparts for comparison.

In [1]:
import torch
import torchvision.transforms.v2 as tv
import torchvision.transforms.v2.functional as tv_fn
from PIL import Image
import numpy as np

# DALI TorchVision API — object-oriented
import nvidia.dali.experimental.torchvision as dtv

# DALI TorchVision API — functional
import nvidia.dali.experimental.torchvision.v2.functional as dtv_fn

print("DALI operators available:", dtv.__all__)
print("DALI functional ops available:", dtv_fn.__all__)

DALI operators available: ['CenterCrop', 'ColorJitter', 'Compose', 'GaussianBlur', 'Grayscale', 'InterpolationMode', 'Normalize', 'Pad', 'PILToTensor', 'RandomApply', 'RandomCrop', 'RandomGrayscale', 'RandomHorizontalFlip', 'RandomResizedCrop', 'RandomVerticalFlip', 'Resize', 'ToPILImage', 'ToPureTensor']
DALI functional ops available: ['center_crop', 'crop', 'gaussian_blur', 'get_dimensions', 'get_image_size', 'get_size', 'horizontal_flip', 'normalize', 'pad', 'pil_to_tensor', 'resize', 'resized_crop', 'rgb_to_grayscale', 'to_grayscale', 'to_pil_image', 'to_tensor', 'vertical_flip']


In [2]:
# A sample image, as a CHW uint8 tensor and as a PIL Image
ramp = torch.linspace(0, 255, 64)
tensor_img = torch.stack(
    [
        ramp.expand(64, 64),
        ramp[:, None].expand(64, 64),
        (ramp + ramp[:, None]) / 2,
    ]
).to(torch.uint8)
pil_img = Image.fromarray(tensor_img.permute(1, 2, 0).numpy())

## PIL Image input

In [3]:
# Build a DALI pipeline that resizes and pads
pipeline_pil = dtv.Compose(
    [
        dtv.Resize(size=(32, 32)),
        dtv.Pad(padding=4, padding_mode="reflect"),
    ]
)

out_dali = pipeline_pil(pil_img)
print("DALI output type:", type(out_dali), "size:", out_dali.size)

# Equivalent TorchVision pipeline for comparison
pipeline_tv = tv.Compose(
    [
        tv.Resize(size=(32, 32)),
        tv.Pad(padding=4, padding_mode="reflect"),
    ]
)

out_tv = pipeline_tv(pil_img)
print("TV   output type:", type(out_tv), "size:", out_tv.size)

# Verify pixel-level equality — cast to float since torch.allclose requires floating-point tensors
dali_t = tv_fn.pil_to_tensor(out_dali).float()
tv_t = tv_fn.pil_to_tensor(out_tv).float()
print("Outputs are equal:", torch.allclose(dali_t, tv_t, rtol=0, atol=1))

DALI output type: <class 'PIL.Image.Image'> size: (40, 40)
TV   output type: <class 'PIL.Image.Image'> size: (40, 40)
Outputs are equal: True


## torch.Tensor input

Pass a `torch.Tensor` of shape `(..., C, H, W)` directly — `Compose` infers the layout from the number
of dimensions (the 3-D tensor below is `CHW`) and returns a `torch.Tensor` of the same shape convention.

In [4]:
pipeline_tensor = dtv.Compose(
    [
        dtv.CenterCrop(size=(48, 48)),
        dtv.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Normalize expects float input in [0, 1] range; scale from [0, 255]
tensor_img_01 = tensor_img / 255.0
out_dali_t = pipeline_tensor(tensor_img_01)
print("DALI output shape:", out_dali_t.shape, "dtype:", out_dali_t.dtype)

pipeline_tv_t = tv.Compose(
    [
        tv.CenterCrop(size=(48, 48)),
        tv.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

out_tv_t = pipeline_tv_t(tensor_img_01)
print("TV   output shape:", out_tv_t.shape, "dtype:", out_tv_t.dtype)
print("Max absolute diff:", (out_dali_t - out_tv_t).abs().max().item())

DALI output shape: torch.Size([3, 48, 48]) dtype: torch.float32
TV   output shape: torch.Size([3, 48, 48]) dtype: torch.float32
Max absolute diff: 0.0
